# L-shaped plate with holes — multipatch NURBS construction

Building an L-shaped domain, each arm perforated with circular holes, as a NURBS
multipatch assembly: diagonal-cut squares (4 patches each) for the straight arms,
joined at the inner corner by two curved sweep patches.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from yeti_iga.future.bspline import (
    BSpline, BSplineSurface, ControlPointManager,
    Patch, GlobalDOFManager, PatchDOFManager, PatchAssembly,
)
from yeti_iga.future.postprocessing.plotting import plot_patches_2d
from yeti_iga.future.refinement import refine_from

---

## L-shape — two squares + two corner patches + vertical arm

Two corner patches are attached to the **left** of square A, together filling the angular
region between the left side of A (x = −1) and the outer arc (centre C = (−1, −2), radius 1).
Two more squares (C, D) then form the **vertical arm** below the corner.

### Corner patch 1

Corners: CP7=(−1,−1), CP6=(−1,1), P2=(−4,1), D=(−1−sin45°, −2+cos45°).  
**Bottom edge** CP7→D: 45° CCW arc, centre C=(−1,−2), from 90° to 135°.  
Arc apex = (−√2, −1), weight = cos(π/8).

| k=u+3v | u=0 (shared with A) | u=1 | u=2 (outer) |
|---|---|---|---|
| v=0 (bottom arc) | CP7 = (−1,−1) | (−√2,−1) w=cos(π/8) | D |
| v=1 | CP13 = (−1,0) | interior (Coons) | mid(D, P2) |
| v=2 (top) | CP6 = (−1,1) | (−2.5,1) | P2=(−4,1) |

Interface with A's patch 2: u=0 boundary ↔ v=1 boundary, `reversed=True`.

### Corner patch 2 — symmetric to patch 1 about the line P2–D

The line from P2=(−4,1) to D lies on the boundary shared by the two corner patches.
Reflecting corner patch 1 about this axis maps CP7→(−2,−2) and CP6→(−4,−2),
while P2 and D stay fixed (they are on the axis).

Corners: D, P2=(−4,1), (−4,−2), (−2,−2).  
**Left edge** D→(−2,−2): 45° CCW arc, same centre C=(−1,−2), from 135° to 180°.  
Arc apex = (−2, −3+√2), weight = cos(π/8).

| k=u+3v | u=0 (arc) | u=1 | u=2 (outer) |
|---|---|---|---|
| v=0 (shared with patch 1) | D | mid(D, P2) | P2=(−4,1) |
| v=1 | (−2,−3+√2) w=cos(π/8) | interior (Coons) | (−4,−0.5) |
| v=2 (bottom) | (−2,−2) | (−3,−2) | (−4,−2) |

Interface with corner patch 1: v=0 boundary ↔ u=2 boundary, same order → `reversed=False`.

### Vertical arm — squares C and D

Corner patch 2's bottom edge (v=2), from (−2,−2) to (−4,−2), is 2 units wide — the same
width as square A's side. Two more diagonal-cut squares are stacked below it, each reusing
3 CPs from the patch above:

- **Square C**: centre (−3,−3), hole at (−3,−3). Its outer top edge reuses
  CP36=(−2,−2), CP39=(−3,−2), CP40=(−4,−2) from corner patch 2.
- **Square D**: centre (−3,−5), hole at (−3,−5). Its outer top edge reuses
  CP45=(−2,−4), CP53=(−3,−4), CP46=(−4,−4) from square C.

Both are structured exactly like squares A/B (diagonal-cut, 4 patches, shape `[3,2]`).

**Interface orientations** (same alternation as A↔B):
- corner patch 2 (v=2) ↔ C patch1 (v=1): same CP order → `reversed=False`
- C patch3 (v=1) ↔ D patch1 (v=1): CPs reversed → `reversed=True`

In [ ]:
R  = 0.25;  L  = 1.0
w_c = 1.0 / np.sqrt(2.0)
r_d = R / np.sqrt(2.0)
r_a = R * np.sqrt(2.0)

la_mgr = ControlPointManager(dim=2)

# ── Square A: center (0, 0) ──────────────────────────────────────────────────
la_mgr.add_point([ r_d, -r_d], w=1.0)   # CP  0 — A inner θ=−45°
la_mgr.add_point([ r_d,  r_d], w=1.0)   # CP  1 — A inner θ=+45°
la_mgr.add_point([-r_d,  r_d], w=1.0)   # CP  2 — A inner θ=+135°
la_mgr.add_point([-r_d, -r_d], w=1.0)   # CP  3 — A inner θ=+225°
la_mgr.add_point([ L,   -L  ], w=1.0)   # CP  4 — A corner ( L,-L), shared A/B
la_mgr.add_point([ L,    L  ], w=1.0)   # CP  5 — A corner ( L, L), shared A/B
la_mgr.add_point([-L,    L  ], w=1.0)   # CP  6 — A corner (-L, L), shared A/corner1
la_mgr.add_point([-L,   -L  ], w=1.0)   # CP  7 — A corner (-L,-L), shared A/corner1
la_mgr.add_point([ r_a,  0.0], w=w_c)   # CP  8 — A patch0 arc apex
la_mgr.add_point([ L,    0.0], w=1.0)   # CP  9 — A outer mid x=L, shared A/B
la_mgr.add_point([ 0.0,  r_a], w=w_c)   # CP 10 — A patch1 arc apex
la_mgr.add_point([ 0.0,  L  ], w=1.0)   # CP 11 — A patch1 outer mid
la_mgr.add_point([-r_a,  0.0], w=w_c)   # CP 12 — A patch2 arc apex
la_mgr.add_point([-L,    0.0], w=1.0)   # CP 13 — A outer mid x=−L, shared A/corner1
la_mgr.add_point([ 0.0, -r_a], w=w_c)   # CP 14 — A patch3 arc apex
la_mgr.add_point([ 0.0, -L  ], w=1.0)   # CP 15 — A patch3 outer mid

# ── Square B: center (2L, 0) — 13 new; CPs 4, 5, 9 reused from A ─────────────
cx = 2*L
la_mgr.add_point([cx+r_d, -r_d], w=1.0)  # CP 16 — B inner θ=−45°
la_mgr.add_point([cx+r_d,  r_d], w=1.0)  # CP 17 — B inner θ=+45°
la_mgr.add_point([cx-r_d,  r_d], w=1.0)  # CP 18 — B inner θ=+135°
la_mgr.add_point([cx-r_d, -r_d], w=1.0)  # CP 19 — B inner θ=+225°
la_mgr.add_point([ 3*L,   -L  ], w=1.0)  # CP 20 — B corner (3L,-L)
la_mgr.add_point([ 3*L,    L  ], w=1.0)  # CP 21 — B corner (3L, L)
la_mgr.add_point([cx+r_a,  0.0], w=w_c)  # CP 22 — B patch0 arc apex
la_mgr.add_point([ 3*L,    0.0], w=1.0)  # CP 23 — B patch0 outer mid
la_mgr.add_point([ cx,     r_a], w=w_c)  # CP 24 — B patch1 arc apex
la_mgr.add_point([ cx,     L  ], w=1.0)  # CP 25 — B patch1 outer mid
la_mgr.add_point([cx-r_a,  0.0], w=w_c)  # CP 26 — B patch2 arc apex
la_mgr.add_point([ cx,    -r_a], w=w_c)  # CP 27 — B patch3 arc apex
la_mgr.add_point([ cx,    -L  ], w=1.0)  # CP 28 — B patch3 outer mid

# ── Corner patch 1 — 6 new CPs (CPs 6, 7, 13 reused from A) ──────────────────
# Bottom edge (v=0): 45° CCW arc from CP7=(-1,-1) to D, centre C=(-1,-2), 90°→135°
# Arc apex at tangent intersection = (-√2, -1), weight = cos(π/8)
w_arc45  = np.cos(np.pi / 8)
D        = np.array([-1.0 - np.sin(np.pi/4), -2.0 + np.cos(np.pi/4)])
arc_apex1 = np.array([-np.sqrt(2.0), -1.0])
P2       = np.array([-4.0,  1.0])
top_mid  = np.array([-2.5,  1.0])            # midpoint CP6–P2 (top straight edge)
left_mid = 0.5 * (D + P2)                    # midpoint D–P2 (outer straight edge)

interior1 = (0.5 * (np.array([-1., 0.]) + left_mid)
           + 0.5 * (arc_apex1 + top_mid)
           - 0.25 * (np.array([-1.,-1.]) + np.array([-1., 1.]) + D + P2))

la_mgr.add_point(arc_apex1.tolist(), w=w_arc45)  # CP 29 — arc apex,   w=cos(π/8)
la_mgr.add_point(D.tolist(),         w=1.0)       # CP 30 — arc end D,  shared c1/c2
la_mgr.add_point(left_mid.tolist(),  w=1.0)       # CP 31 — outer-edge mid, shared c1/c2
la_mgr.add_point(interior1.tolist(), w=1.0)       # CP 32 — interior c1
la_mgr.add_point(top_mid.tolist(),   w=1.0)       # CP 33 — top-edge mid
la_mgr.add_point(P2.tolist(),        w=1.0)       # CP 34 — top-left corner, shared c1/c2

# ── Corner patch 2 — 6 new CPs (CPs 30, 31, 34 reused from corner patch 1) ───
# Left edge (u=0): 45° CCW arc from D to (-2,-2), centre C=(-1,-2), 135°→180°
# Tangent at D (135°): (−1/√2, −1/√2);  tangent at (−2,−2) (180°): (0, −1)
# Arc apex = (−2, −3+√2), weight = cos(π/8)
arc_apex2  = np.array([-2.0, -3.0 + np.sqrt(2.0)])
bot_R      = np.array([-2.0, -2.0])              # bottom-right corner
mid_right  = 0.5 * (P2 + np.array([-4., -2.]))   # midpoint P2–(-4,-2)  on u=2 edge
bot_L      = np.array([-4.0, -2.0])              # bottom-left corner
mid_bot    = np.array([-3.0, -2.0])              # midpoint of bottom straight edge

interior2 = (0.5 * (arc_apex2 + mid_right)
           + 0.5 * (left_mid + mid_bot)
           - 0.25 * (D + bot_R + P2 + bot_L))

la_mgr.add_point(arc_apex2.tolist(), w=w_arc45)  # CP 35 — arc apex,   w=cos(π/8)
la_mgr.add_point(bot_R.tolist(),     w=1.0)       # CP 36 — (-2,-2), shared corner2/C
la_mgr.add_point(mid_right.tolist(), w=1.0)       # CP 37 — mid P2–(-4,-2)
la_mgr.add_point(interior2.tolist(), w=1.0)       # CP 38 — interior c2
la_mgr.add_point(mid_bot.tolist(),   w=1.0)       # CP 39 — (-3,-2), shared corner2/C
la_mgr.add_point(bot_L.tolist(),     w=1.0)       # CP 40 — (-4,-2), shared corner2/C

# ── Square C: center (-3, -3) — vertical arm, 13 new; CPs 36, 39, 40 reused ──
cx_c, cy_c = -3.0, -3.0
la_mgr.add_point([cx_c+r_d, cy_c-r_d], w=1.0)  # CP 41 — C inner θ=−45°
la_mgr.add_point([cx_c+r_d, cy_c+r_d], w=1.0)  # CP 42 — C inner θ=+45°
la_mgr.add_point([cx_c-r_d, cy_c+r_d], w=1.0)  # CP 43 — C inner θ=+135°
la_mgr.add_point([cx_c-r_d, cy_c-r_d], w=1.0)  # CP 44 — C inner θ=+225°
la_mgr.add_point([cx_c+L,   cy_c-L  ], w=1.0)  # CP 45 — C corner ( L,-L), shared C/D
la_mgr.add_point([cx_c-L,   cy_c-L  ], w=1.0)  # CP 46 — C corner (-L,-L), shared C/D
la_mgr.add_point([cx_c+r_a, cy_c    ], w=w_c)  # CP 47 — C patch0 arc apex
la_mgr.add_point([cx_c+L,   cy_c    ], w=1.0)  # CP 48 — C patch0 outer mid
la_mgr.add_point([cx_c,     cy_c+r_a], w=w_c)  # CP 49 — C patch1 arc apex
la_mgr.add_point([cx_c-r_a, cy_c    ], w=w_c)  # CP 50 — C patch2 arc apex
la_mgr.add_point([cx_c-L,   cy_c    ], w=1.0)  # CP 51 — C patch2 outer mid
la_mgr.add_point([cx_c,     cy_c-r_a], w=w_c)  # CP 52 — C patch3 arc apex
la_mgr.add_point([cx_c,     cy_c-L  ], w=1.0)  # CP 53 — C outer mid, shared C/D

# ── Square D: center (-3, -5) — vertical arm, 13 new; CPs 45, 46, 53 reused ──
cx_d, cy_d = -3.0, -5.0
la_mgr.add_point([cx_d+r_d, cy_d-r_d], w=1.0)  # CP 54 — D inner θ=−45°
la_mgr.add_point([cx_d+r_d, cy_d+r_d], w=1.0)  # CP 55 — D inner θ=+45°
la_mgr.add_point([cx_d-r_d, cy_d+r_d], w=1.0)  # CP 56 — D inner θ=+135°
la_mgr.add_point([cx_d-r_d, cy_d-r_d], w=1.0)  # CP 57 — D inner θ=+225°
la_mgr.add_point([cx_d+L,   cy_d-L  ], w=1.0)  # CP 58 — D corner ( L,-L)
la_mgr.add_point([cx_d-L,   cy_d-L  ], w=1.0)  # CP 59 — D corner (-L,-L)
la_mgr.add_point([cx_d+r_a, cy_d    ], w=w_c)  # CP 60 — D patch0 arc apex
la_mgr.add_point([cx_d+L,   cy_d    ], w=1.0)  # CP 61 — D patch0 outer mid
la_mgr.add_point([cx_d,     cy_d+r_a], w=w_c)  # CP 62 — D patch1 arc apex
la_mgr.add_point([cx_d-r_a, cy_d    ], w=w_c)  # CP 63 — D patch2 arc apex
la_mgr.add_point([cx_d-L,   cy_d    ], w=1.0)  # CP 64 — D patch2 outer mid
la_mgr.add_point([cx_d,     cy_d-r_a], w=w_c)  # CP 65 — D patch3 arc apex
la_mgr.add_point([cx_d,     cy_d-L  ], w=1.0)  # CP 66 — D outer mid bottom

print(f'la_mgr.n_points = {la_mgr.n_points}  (expected 67)')
print(f'la_mgr.is_rational = {la_mgr.is_rational}')
print(f'D         = {[round(c,4) for c in D.tolist()]}')
print(f'arc_apex1 = {[round(c,4) for c in arc_apex1.tolist()]}  w={round(w_arc45,4)}')
print(f'arc_apex2 = {[round(c,4) for c in arc_apex2.tolist()]}  w={round(w_arc45,4)}')
print(f'interior1 = {[round(c,4) for c in interior1.tolist()]}')
print(f'interior2 = {[round(c,4) for c in interior2.tolist()]}')

# ── Patch surfaces ────────────────────────────────────────────────────────────
def make_la_surface():
    return BSplineSurface(BSpline(2, np.array([0., 0., 0., 1., 1., 1.])),
                          BSpline(1, np.array([0., 0., 1., 1.])))

def make_corner_surface():
    return BSplineSurface(BSpline(2, np.array([0., 0., 0., 1., 1., 1.])),
                          BSpline(2, np.array([0., 0., 0., 1., 1., 1.])))

la_all_maps = [
    # Square A (shape [3,2])
    [0,  8,  1,  4,  9,  5],
    [1, 10,  2,  5, 11,  6],
    [2, 12,  3,  6, 13,  7],   # A patch2 — v=1 outer edge shared with corner 1
    [3, 14,  0,  7, 15,  4],
    # Square B (shape [3,2])
    [16, 22, 17, 20, 23, 21],
    [17, 24, 18, 21, 25,  5],
    [18, 26, 19,  5,  9,  4],
    [19, 27, 16,  4, 28, 20],
    # Corner 1 (shape [3,3]): k = u_idx + 3*v_idx
    #   v=0 (bottom arc):  [CP7=7,  arc_apex1=29, D=30   ]
    #   v=1 (middle):      [CP13=13, interior1=32, left_mid=31]
    #   v=2 (top):         [CP6=6,  top_mid=33,  P2=34  ]
    [7, 29, 30,  13, 32, 31,  6, 33, 34],
    # Corner 2 (shape [3,3]): k = u_idx + 3*v_idx
    #   v=0 (shared with c1, u varies): [D=30,      left_mid=31, P2=34  ]
    #   v=1 (middle):                   [arc_apex2=35, interior2=38, mid_right=37]
    #   v=2 (bottom straight):          [bot_R=36,  mid_bot=39,  bot_L=40]
    [30, 31, 34,  35, 38, 37,  36, 39, 40],
    # Square C (shape [3,2]) — outer top edge (v=1) reuses CPs 36, 39, 40 from corner 2
    [41, 47, 42, 45, 48, 36],  # C patch0 right
    [42, 49, 43, 36, 39, 40],  # C patch1 top    — shared with corner 2 (reversed=False)
    [43, 50, 44, 40, 51, 46],  # C patch2 left
    [44, 52, 41, 46, 53, 45],  # C patch3 bottom
    # Square D (shape [3,2]) — outer top edge (v=1) reuses CPs 45, 46, 53 from square C
    [54, 60, 55, 58, 61, 45],  # D patch0 right
    [55, 62, 56, 45, 53, 46],  # D patch1 top    — shared with C patch3 (reversed=True)
    [56, 63, 57, 46, 64, 59],  # D patch2 left
    [57, 65, 54, 59, 66, 58],  # D patch3 bottom
]
la_shapes = [[3, 2]] * 8 + [[3, 3], [3, 3]] + [[3, 2]] * 8

la_dofs = 2
la_gm   = GlobalDOFManager([la_dofs] * la_mgr.n_points)
la_patches = []
for m, sh in zip(la_all_maps, la_shapes):
    surf = make_corner_surface() if len(m) == 9 else make_la_surface()
    pdm  = PatchDOFManager(la_dofs, m, la_gm)
    la_patches.append(Patch(surf, la_mgr, m, sh, pdm))

print(f'n_cp per patch: {[p.n_cp for p in la_patches]}')

In [ ]:
la_assembly = PatchAssembly()
for p in la_patches:
    la_assembly.add_patch(p)

la_assembly.detect_shared_control_points()
la_shared = la_assembly.get_shared_control_points_map()

coords = la_mgr.coords_view()
print(f'Shared CPs ({len(la_shared)} total):')
for cp_id, pidx in la_shared.items():
    xy = [round(c, 4) for c in coords[cp_id].tolist()]
    print(f'  CP {cp_id:2d} {str(xy):28s} patches {pidx}')

la_assembly.detect_interfaces()
print('\nDetected interfaces:')
for itf in la_assembly.get_interfaces():
    print(f'  patch {itf["patch_a"]:2d} (dir={itf["direction_a"]}, side={itf["side_a"]}, '
          f'vdir={itf["varying_direction_a"]}) ↔ '
          f'patch {itf["patch_b"]:2d} (dir={itf["direction_b"]}, side={itf["side_b"]}, '
          f'vdir={itf["varying_direction_b"]}) reversed={itf["reversed"]}')

plot_patches_2d(la_assembly, show_control_points=True, show_control_point_indices=True,
                fill_patches=True, fill_alpha=0.25, legend=True,
                title='L-shape — squares A, B, C, D + 2 corner patches (18 patches)')

---

## Local refinement with multi-hop propagation

`PatchAssembly.refine_with_propagation(patch_index, direction, refine_1d_fn)` refines one
patch along `direction` and propagates to its **direct** neighbors only (one hop): any
interface whose *varying direction* (the direction the shared edge itself runs along)
matches `direction` gets its neighbor refined too, using whichever direction the C++ side
determines is the matching one on that neighbor (handling crossed interfaces automatically).

For a refinement that needs to travel further than one hop — e.g. all the way around a
loop of quarter-patches — `refine_from()` (`yeti_iga.future.refinement`) repeatedly
re-triggers propagation from each newly-touched patch (using the direction that was
actually applied to it, captured by a guarded callback) until no new patch is reached. Its
`refiner_factory` argument makes the kind of refinement pluggable — uniform h-refinement
(`SubdivisionRefiner`, the default used below), degree elevation (`PRefiner`), or a single
targeted knot insertion (`HRefiner`) all expose the same `refine_1d(patch, protected_ids)`
interface.

Refining direction *d* only adds new control points to the edges whose own parameter runs
along *d* (its `varying_direction`); the two edges held fixed at a *d*-boundary are
untouched. For a quarter-patch, degree-2 direction **u is angular** and degree-1 direction
**v is radial**. Both of `u=0`/`u=2` (the radial spokes) are perpendicular to the horizontal
inner-arc/outer-edge boundaries, so refining **u adds points along those spokes** — a
**vertical** effect for the top/bottom quarters — while refining **v adds points along the
horizontal inner arc / outer edge** — the **horizontal** effect.

**Two exercises on the L-shape:**
1. **Patch 17** (square D, bottom quarter), direction 1 (v, radial → horizontal effect for
   this quarter, since it thickens the horizontal inner-arc/outer-edge boundaries). Its
   radial spokes (shared with patches 14 and 16, `varying_direction=1`) propagate the
   refinement **around the full loop** of square D's four quarter-patches
   (14, 15, 16, 17).
2. **Patch 15** (square D, top quarter), direction 0 (u, angular → vertical effect,
   thickening the radial spokes). Its outer edge at y = −4 is shared with square C's
   bottom quarter (patch 13) with `varying_direction=0` → propagates **once, into patch 13**
   only (patch 13's other interfaces have `varying_direction=1`, so it stops there).

In [ ]:
# Exercise 1: patch 17 (D bottom), direction 1 (v, radial -> horizontal effect)
# Expect propagation around the full loop of square D (14, 15, 16, 17)
n_cp_before = [p.n_cp for p in la_patches]

touched1 = refine_from(la_assembly, la_patches, start_index=17, start_direction=1)

print('Patches touched:', touched1)
for i, (b, a) in enumerate(zip(n_cp_before, [p.n_cp for p in la_patches])):
    if b != a:
        print(f'  patch {i:2d}: n_cp {b} -> {a}')
print(f'la_mgr.n_points: {la_mgr.n_points}')

In [ ]:
# Exercise 2: patch 15 (D top), direction 0 (u, angular -> vertical effect)
# Expect propagation once, into square C's bottom quarter (patch 13)
n_cp_before = [p.n_cp for p in la_patches]

touched2 = refine_from(la_assembly, la_patches, start_index=15, start_direction=0)

print('Patches touched:', touched2)
for i, (b, a) in enumerate(zip(n_cp_before, [p.n_cp for p in la_patches])):
    if b != a:
        print(f'  patch {i:2d}: n_cp {b} -> {a}')
print(f'la_mgr.n_points: {la_mgr.n_points}')

In [ ]:
plot_patches_2d(la_assembly, show_control_points=True, show_control_point_indices=False,
                fill_patches=True, fill_alpha=0.25, legend=True,
                title='After refining patch 17 (v, horiz.) and patch 15 (u, vert.)')